# Chapter 22: Cayley-Klein Geometries at Work

**Source orientation.** Sections 22.1-22.5; printed pages 423-442; PDF pages 445-464.

**Chapter goal.** Build elementary geometry from a projective absolute: perpendiculars, altitudes, midpoints, angle bisectors, and trigonometric ratios should be computed from the fundamental conic rather than assumed from Euclidean intuition.

The chapter's useful warning is that a Cayley-Klein theorem often has two versions. A constructive version names the polar, pole, midpoint, or bisector explicitly and usually has a clean algebraic check. An implicit version gives only an equation such as $l^T Bm=0$ and can become underdetermined when the absolute degenerates. The visuals below keep that distinction visible.

In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate the Perspectives-on-Projective-Geometry course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

CHAPTER_SLUG = "chapter-22-cayley-klein-geometries-at-work"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / CHAPTER_SLUG
FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
TABLES = ARTIFACT_ROOT / "tables"
CHECKS = ARTIFACT_ROOT / "checks"
for folder in [FIGURES, HTML, TABLES, CHECKS]:
    folder.mkdir(parents=True, exist_ok=True)

print(BOOK_ROOT.relative_to(BOOK_ROOT.parent))

## Computational Translation Guide

A Cayley-Klein model is a projective plane plus an absolute conic. The notebook uses this dictionary.

| Book object | Computation in this notebook | What gets checked |
| --- | --- | --- |
| point $p$ and line $l$ | homogeneous vectors in $\mathbb{R}^3$ | incidence $p^T l = 0$ |
| primal conic $A$ | symmetric matrix with $p^T A p=0$ on the absolute | point measurements and midpoint formula |
| dual conic $B$ | symmetric matrix with $l^T B l=0$ for tangent lines | angle measurements and orthogonality |
| pole of a line | $B l$, when nonzero | a perpendicular through $p$ is `join(p, B l)` |
| orthogonality | $l^T Bm=0$ | altitude residuals and line concurrence |
| midpoint pair | $\sqrt{\Omega_{qq}}p \pm \sqrt{\Omega_{pp}}q$ | harmonic cross-ratio $-1$ |
| angle-bisector pair | $\sqrt{\Theta_{ll}}g \pm \sqrt{\Theta_{gg}}l$ | the two bisectors are $B$-orthogonal |
| trigonometric sine-law shadow | ratios built from $\Psi$ and $\Psi^*$ | projective ratios agree numerically |

**Library routing.** Matplotlib is used for durable 2D construction diagrams because the chapter is about lines, conics, poles, and incidences. Plotly is used for the parameter lab because the law-of-sines identity is easier to inspect while moving a vertex. SymPy records the short symbolic cancellations behind midpoint harmonicity and the final trigonometric proof step. NetworkX is used for a proof-dependency map, not as decoration, so the learner can see which constructions feed which theorems.

In [ ]:
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import Markdown

from utils.artifacts import assert_artifacts, book_relative, display_artifact, image_stats, save_json, save_table
from utils.projective import cross_ratio, hpoint, join, meet

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10})

# Unit circle x^2 + y^2 = z^2, with the sign that makes p^T A p
# positive for points inside the disk.
A_UNIT = np.diag([-1.0, -1.0, 1.0])
B_UNIT = np.diag([-1.0, -1.0, 1.0])


def hnormalize(point, tol=1e-12):
    p = np.asarray(point, dtype=float)
    if abs(p[2]) < tol:
        return p.copy()
    return p / p[2]


def normalize_line(line, tol=1e-12):
    l = np.asarray(line, dtype=float)
    scale = np.linalg.norm(l[:2])
    if scale < tol:
        return l.copy()
    return l / scale


def line_points(line, xlim=(-1.7, 1.7), ylim=(-1.7, 1.7), tol=1e-10):
    a, b, c = np.asarray(line, dtype=float)
    candidates = []
    if abs(b) > tol:
        for x in xlim:
            y = -(a * x + c) / b
            if ylim[0] - 1e-8 <= y <= ylim[1] + 1e-8:
                candidates.append((x, y))
    if abs(a) > tol:
        for y in ylim:
            x = -(b * y + c) / a
            if xlim[0] - 1e-8 <= x <= xlim[1] + 1e-8:
                candidates.append((x, y))
    unique = []
    for candidate in candidates:
        if not any(np.linalg.norm(np.asarray(candidate) - np.asarray(old)) < 1e-7 for old in unique):
            unique.append(candidate)
    if len(unique) < 2:
        return np.empty((0, 2))
    if len(unique) == 2:
        return np.asarray(unique)
    pairs = []
    for i, p in enumerate(unique):
        for q in unique[i + 1:]:
            pairs.append((np.linalg.norm(np.asarray(p) - np.asarray(q)), p, q))
    _, p, q = max(pairs, key=lambda item: item[0])
    return np.asarray([p, q])


def draw_line(ax, line, *, xlim=(-1.7, 1.7), ylim=(-1.7, 1.7), label=None, color="black", lw=1.8, ls="-", alpha=1.0):
    pts = line_points(line, xlim=xlim, ylim=ylim)
    if len(pts) == 2:
        ax.plot(pts[:, 0], pts[:, 1], color=color, lw=lw, ls=ls, alpha=alpha, label=label)


def draw_point(ax, point, *, label=None, color="black", marker="o", size=46, dx=0.03, dy=0.03):
    p = hnormalize(point)
    ax.scatter([p[0]], [p[1]], s=size, color=color, marker=marker, zorder=5)
    if label:
        ax.text(p[0] + dx, p[1] + dy, label, color=color, fontsize=9)


def draw_unit_circle(ax, *, color="#333333", lw=1.6, label="absolute"):
    theta = np.linspace(0, 2 * np.pi, 500)
    ax.plot(np.cos(theta), np.sin(theta), color=color, lw=lw, label=label)


def set_equal_window(ax, xlim=(-1.35, 1.35), ylim=(-1.35, 1.35), title=None):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.axhline(0, color="#dddddd", lw=0.8)
    ax.axvline(0, color="#dddddd", lw=0.8)
    if title:
        ax.set_title(title, fontsize=11)


def save_figure(fig, filename):
    path = FIGURES / filename
    fig.savefig(path, dpi=165, bbox_inches="tight")
    plt.close(fig)
    return path


def quadratic_form(matrix, vector):
    v = np.asarray(vector, dtype=float)
    return float(v @ matrix @ v)


def omega(point, A=A_UNIT):
    return quadratic_form(A, point)


def theta(line, B=B_UNIT):
    return quadratic_form(B, line)


def pole(line, B=B_UNIT):
    return B @ np.asarray(line, dtype=float)


def ck_orthogonality(line_a, line_b, B=B_UNIT):
    return float(np.asarray(line_a, dtype=float) @ B @ np.asarray(line_b, dtype=float))


def psi_point(P, Q, A=A_UNIT, B=B_UNIT):
    side = join(P, Q)
    return float((side @ B @ side) / ((P @ A @ P) * (Q @ A @ Q)))


def psi_line(line_a, line_b, A=A_UNIT, B=B_UNIT):
    vertex = meet(line_a, line_b)
    return float((vertex @ A @ vertex) / ((line_a @ B @ line_a) * (line_b @ B @ line_b)))


def line_through_point_direction(point, direction):
    p = hnormalize(point)
    q = p + np.array([direction[0], direction[1], 0.0])
    return join(p, q)


def all_book_relative(paths):
    return [book_relative(path) for path in paths]

In [ ]:
STORYBOARD = {
    "chapter goal": "Compute familiar elementary constructions from a projective absolute conic and see where degeneracy changes the theorem.",
    "source span read": "Perspectives on Projective Geometry, Chapter 22, sections 22.1-22.5; printed pages 423-442; PDF pages 445-464",
    "concept inventory": [
        "orthogonality as l^T Bm = 0 and as a pole-polar incidence",
        "constructive altitude theorem versus implicit orthogonality constraints",
        "Euclidean, pseudo-Euclidean, rank-one, and hyperbolic behavior compared from the same equation",
        "two Cayley-Klein midpoints and their harmonic relation",
        "dual angle-bisector formula and bisector orthogonality",
        "projective law of sines through Psi and Psi-star ratios",
    ],
    "library routing table": [
        {"concept": "pole-polar perpendiculars", "representation": "labeled conic and altitude diagram", "library": "Matplotlib + NumPy", "why": "incidence and line/conic structure are two-dimensional and need durable labels"},
        {"concept": "implicit degeneracy", "representation": "rank-one dual conic counterexample", "library": "Matplotlib + NumPy", "why": "the failure is a visible choice of nonconcurrent lines satisfying l^TBg=0"},
        {"concept": "model comparison", "representation": "slope panels and right hyperbolic pentagon", "library": "Matplotlib + Pandas", "why": "side-by-side static comparison and a tabular check expose shared and differing behavior"},
        {"concept": "proof dependencies", "representation": "directed dependency graph", "library": "NetworkX", "why": "the chapter's statements build from a small chain of projective translations"},
        {"concept": "midpoints and angle bisectors", "representation": "formula-driven construction diagram", "library": "Matplotlib + SymPy", "why": "the visual pair needs exact harmonic and bilinear checks nearby"},
        {"concept": "Cayley-Klein trigonometry", "representation": "interactive moving-vertex law-of-sines lab", "library": "Plotly + NumPy", "why": "the invariant is best inspected while a triangle changes"},
    ],
    "visual sequence": [
        {"artifact_filename": "orthogonality-pole-polar-altitudes.png", "concept": "orthogonality from pole-polar relation", "inspection_target": "each altitude passes through the pole of the opposite side", "validation": "l_i^T B h_i and altitude concurrence residuals"},
        {"artifact_filename": "constructive-vs-implicit-altitudes.png", "concept": "constructive versus implicit representations", "inspection_target": "rank-one B makes two altitude choices free", "validation": "chosen lines satisfy l_i^T B g_i=0 but are not concurrent"},
        {"artifact_filename": "cayley-klein-model-comparison.png", "concept": "shared and differing Cayley-Klein properties", "inspection_target": "Euclidean slope product, pseudo-Euclidean inverse slope, and hyperbolic right pentagon", "validation": "slope and adjacent-line orthogonality residuals"},
        {"artifact_filename": "midpoints-angle-bisectors-duality.png", "concept": "midpoints and angle bisectors", "inspection_target": "one midpoint lies between p and q while the harmonic mate may sit outside the absolute", "validation": "cross-ratio -1 and bisector B-orthogonality"},
        {"artifact_filename": "projective-law-of-sines-check.png", "concept": "trigonometry", "inspection_target": "three Psi/Psi-star fractions attached to one triangle", "validation": "ratio spread below tolerance"},
        {"artifact_filename": "projective-law-of-sines-lab.html", "concept": "Cayley-Klein trigonometry lab", "inspection_target": "hover over moving vertices and compare the invariant ratio", "validation": "CSV sweep and maximum ratio spread"},
    ],
    "artifact plan": {
        "figures": [
            "figures/orthogonality-pole-polar-altitudes.png",
            "figures/constructive-vs-implicit-altitudes.png",
            "figures/cayley-klein-model-comparison.png",
            "figures/midpoints-angle-bisectors-duality.png",
            "figures/projective-law-of-sines-check.png",
            "figures/chapter-22-proof-dependency-map.png",
        ],
        "html": ["html/projective-law-of-sines-lab.html"],
        "tables": ["tables/model-comparison-samples.csv", "tables/law-of-sines-sweep.csv"],
        "checks": ["checks/storyboard.json", "checks/visual-checks.json", "checks/final-sanity.json", "checks/symbolic-identities.json"],
    },
    "computational checks": [
        "pole-polar altitude residuals",
        "constructive/implicit degeneracy residuals and nonconcurrence",
        "right-pentagon adjacent orthogonality",
        "midpoint harmonic cross-ratio",
        "angle-bisector orthogonality",
        "projective law-of-sines ratio equality for a concrete triangle and for a sweep",
        "artifact existence, nonzero file sizes, and nonblank image statistics",
    ],
    "proof-visualization strategy": "Use a NetworkX dependency map and small symbolic identities instead of copying proof prose.",
    "risks": "The notebook avoids degenerate edge cases not needed for the chapter argument; all source use is orientation only.",
}
storyboard_path = save_json(STORYBOARD, ARTIFACT_ROOT, "checks", "storyboard.json")
book_relative(storyboard_path)

## 1. Orthogonality From Pole-Polar Relations

The source starts by replacing the Euclidean phrase "right angle" with a projective test. If $B$ is the dual matrix of the absolute conic, then two lines are orthogonal exactly when

$$l^T Bm = 0.$$

When $Bl$ is nonzero, it is the pole of $l$. A perpendicular to $l$ through a point $p$ is therefore the line joining $p$ to $Bl$. The first figure uses this constructive version three times: each side of a triangle gets a pole, and the line from the opposite vertex to that pole is an altitude.

In [ ]:
P = hpoint(-0.48, -0.25)
Q = hpoint(0.62, -0.12)
R = hpoint(0.08, 0.58)

side_p = normalize_line(join(Q, R))
side_q = normalize_line(join(R, P))
side_r = normalize_line(join(P, Q))
sides = {"p=QR": side_p, "q=RP": side_q, "r=PQ": side_r}
vertices = {"P": P, "Q": Q, "R": R}

alt_p = normalize_line(join(P, pole(side_p)))
alt_q = normalize_line(join(Q, pole(side_q)))
alt_r = normalize_line(join(R, pole(side_r)))
altitudes = {"h_P": alt_p, "h_Q": alt_q, "h_R": alt_r}
orthocenter = hnormalize(meet(alt_p, alt_q))

orthogonality_residuals = {
    "p_with_h_P": abs(ck_orthogonality(side_p, alt_p)),
    "q_with_h_Q": abs(ck_orthogonality(side_q, alt_q)),
    "r_with_h_R": abs(ck_orthogonality(side_r, alt_r)),
}
altitude_concurrence_det = abs(float(np.linalg.det(np.vstack([alt_p, alt_q, alt_r]))))
orthocenter_incidence_residual = abs(float(orthocenter @ alt_r))
pole_incidence_residuals = {
    name: abs(float(hnormalize(pole(line)) @ altitude))
    for (name, line), altitude in zip(sides.items(), altitudes.values())
}

fig, ax = plt.subplots(figsize=(7.6, 7.0))
draw_unit_circle(ax)
for label, line in sides.items():
    draw_line(ax, line, xlim=(-1.25, 1.25), ylim=(-1.25, 1.25), label=label, color="#303030", lw=2.0)
for label, line in altitudes.items():
    draw_line(ax, line, xlim=(-1.25, 1.25), ylim=(-1.25, 1.25), label=label, color="#b3261e", lw=1.8, ls="--")
for label, point in vertices.items():
    draw_point(ax, point, label=label, color="#111111", size=58)
for label, line in sides.items():
    draw_point(ax, pole(line), label=label + "*", color="#1d4ed8", marker="s", size=42)
draw_point(ax, orthocenter, label="H", color="#b3261e", marker="D", size=52)
ax.plot([P[0], Q[0], R[0], P[0]], [P[1], Q[1], R[1], P[1]], color="#111111", lw=1.3, alpha=0.6)
set_equal_window(ax, (-1.18, 1.18), (-1.18, 1.18), "Altitudes as joins with side poles")
ax.legend(loc="upper right", fontsize=8, frameon=True)
ax.text(-1.12, -1.13, "$l^T B h=0$ for each side-altitude pair", color="#b3261e", fontsize=10)
orthogonality_path = save_figure(fig, "orthogonality-pole-polar-altitudes.png")

ORTHOGONALITY_CHECKS = {
    "orthogonality_residuals": orthogonality_residuals,
    "altitude_concurrence_det": altitude_concurrence_det,
    "orthocenter_incidence_residual": orthocenter_incidence_residual,
    "pole_incidence_residuals": pole_incidence_residuals,
}
ORTHOGONALITY_CHECKS

## 2. Constructive Versus Implicit Altitudes

The constructive altitude theorem requires the poles $Bl_i$ to exist. The implicit sentence "choose a line $g_i$ through the opposite vertex with $l_i^TBg_i=0$" sounds similar, but it is not equivalent in degenerate models.

The next diagram compares the two modes. On the left, a nondegenerate absolute fixes the altitude because $Bl$ is an actual point. On the right, $B=qq^T$ is a rank-one dual conic. If two side lines pass through $q$, the equations $l_i^TBg_i=0$ impose no restriction on $g_i$. The displayed $g_i$ satisfy the implicit equations but fail to concur.

In [ ]:
constructive_side = side_p
constructive_vertex = P
constructive_pole = pole(constructive_side)
constructive_altitude = alt_p

q0 = hpoint(0.0, 0.0)
B_RANK_ONE = np.outer(q0, q0)
l1 = np.array([1.0, 0.0, 0.0])
l2 = np.array([0.0, 1.0, 0.0])
l3 = normalize_line(np.array([1.0, 1.0, -1.0]))
V1 = hnormalize(meet(l2, l3))
V2 = hnormalize(meet(l3, l1))
V3 = hnormalize(meet(l1, l2))

g1 = normalize_line(line_through_point_direction(V1, (1.0, 0.55)))
g2 = normalize_line(line_through_point_direction(V2, (1.0, -0.35)))
g3 = normalize_line(line_through_point_direction(V3, (1.0, 1.0)))
implicit_residuals = [abs(float(l @ B_RANK_ONE @ g)) for l, g in [(l1, g1), (l2, g2), (l3, g3)]]
implicit_concurrence_det = abs(float(np.linalg.det(np.vstack([g1, g2, g3]))))
rank_one_poles = [np.linalg.norm(B_RANK_ONE @ l) for l in [l1, l2, l3]]

fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.6))
ax = axes[0]
draw_unit_circle(ax)
draw_line(ax, constructive_side, xlim=(-1.15, 1.15), ylim=(-1.15, 1.15), label="side l", color="#303030", lw=2.1)
draw_line(ax, constructive_altitude, xlim=(-1.15, 1.15), ylim=(-1.15, 1.15), label="join(P, Bl)", color="#b3261e", lw=2.0, ls="--")
draw_point(ax, constructive_vertex, label="P", color="#111111", size=58)
draw_point(ax, constructive_pole, label="Bl", color="#1d4ed8", marker="s", size=50)
set_equal_window(ax, (-1.15, 1.15), (-1.15, 1.15), "Constructive: pole fixes the altitude")
ax.legend(loc="lower right", fontsize=8)

ax = axes[1]
for line, label in [(l1, "l1 through q"), (l2, "l2 through q"), (l3, "l3")]:
    draw_line(ax, line, xlim=(-0.35, 1.35), ylim=(-0.35, 1.35), label=label, color="#303030", lw=2.0)
for line, label, color in [(g1, "g1 free", "#b3261e"), (g2, "g2 free", "#d97706"), (g3, "g3 through q", "#7c3aed")]:
    draw_line(ax, line, xlim=(-0.35, 1.35), ylim=(-0.35, 1.35), label=label, color=color, lw=2.0, ls="--")
for point, label in [(V1, "V1"), (V2, "V2"), (V3, "q=V3")]:
    draw_point(ax, point, label=label, color="#111111", size=52)
set_equal_window(ax, (-0.35, 1.35), (-0.35, 1.35), "Implicit rank-one case: equations allow bad choices")
ax.legend(loc="upper right", fontsize=8)
ax.text(-0.31, -0.28, "$l_i^TBg_i=0$ for all three, but $g_1,g_2,g_3$ are not concurrent", fontsize=9, color="#b3261e")
constructive_implicit_path = save_figure(fig, "constructive-vs-implicit-altitudes.png")

CONSTRUCTIVE_IMPLICIT_CHECKS = {
    "rank_one_pole_norms": rank_one_poles,
    "implicit_orthogonality_residuals": implicit_residuals,
    "implicit_concurrence_det": implicit_concurrence_det,
    "interpretation": "nonzero determinant means the displayed implicit altitudes do not share one point",
}
CONSTRUCTIVE_IMPLICIT_CHECKS

## 3. Shared Properties, Different Geometries

The same bilinear test $l^T Bm=0$ produces familiar and unfamiliar pictures depending on the absolute.

Euclidean eyes see perpendicular slopes multiply to $-1$. In the standard pseudo-Euclidean embedding, orthogonal finite lines have inverse slopes, so a line is reflected across a slope-one direction rather than turned through a Euclidean quarter-turn. In the hyperbolic Klein disk, a regular five-sided polygon can have every angle right: adjacent side lines satisfy $l_i^T B l_{i+1}=0$.

In [ ]:
base_slope = 0.55
euclidean_orthogonal_slope = -1 / base_slope
pseudo_orthogonal_slope = 1 / base_slope
slope_checks = {
    "euclidean_product": base_slope * euclidean_orthogonal_slope,
    "pseudo_product": base_slope * pseudo_orthogonal_slope,
}

n = 5
delta = 2 * np.pi / n
rho = math.sqrt(math.cos(delta))
pentagon_lines = [normalize_line(np.array([math.cos(2 * np.pi * k / n), math.sin(2 * np.pi * k / n), -rho])) for k in range(n)]
pentagon_vertices = [hnormalize(meet(pentagon_lines[k], pentagon_lines[(k + 1) % n])) for k in range(n)]
pentagon_residuals = [abs(ck_orthogonality(pentagon_lines[k], pentagon_lines[(k + 1) % n])) for k in range(n)]
pentagon_radii = [float(np.linalg.norm(v[:2])) for v in pentagon_vertices]

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.8))
xx = np.linspace(-1.2, 1.2, 2)
axes[0].plot(xx, base_slope * xx, color="#1d4ed8", lw=2.0, label="slope m")
axes[0].plot(xx, euclidean_orthogonal_slope * xx, color="#b3261e", lw=2.0, ls="--", label="slope -1/m")
set_equal_window(axes[0], (-1.05, 1.05), (-1.05, 1.05), "Euclidean: turn to a negative reciprocal")
axes[0].legend(fontsize=8, loc="upper left")
axes[0].text(-1.0, -0.98, f"m*(-1/m) = {slope_checks['euclidean_product']:.0f}", fontsize=9)

axes[1].plot(xx, base_slope * xx, color="#1d4ed8", lw=2.0, label="slope m")
axes[1].plot(xx, pseudo_orthogonal_slope * xx, color="#b3261e", lw=2.0, ls="--", label="slope 1/m")
axes[1].plot(xx, xx, color="#666666", lw=1.3, ls=":", label="self-orthogonal slope 1")
axes[1].plot(xx, -xx, color="#888888", lw=1.3, ls=":", label="self-orthogonal slope -1")
set_equal_window(axes[1], (-1.05, 1.05), (-1.05, 1.05), "Pseudo-Euclidean: inverse slopes")
axes[1].legend(fontsize=8, loc="upper left")
axes[1].text(-1.0, -0.98, f"m*(1/m) = {slope_checks['pseudo_product']:.0f}", fontsize=9)

ax = axes[2]
draw_unit_circle(ax)
poly = np.array([v[:2] for v in pentagon_vertices] + [pentagon_vertices[0][:2]])
ax.plot(poly[:, 0], poly[:, 1], color="#b3261e", lw=2.2)
ax.fill(poly[:, 0], poly[:, 1], color="#fca5a5", alpha=0.22)
for line in pentagon_lines:
    draw_line(ax, line, xlim=(-1.05, 1.05), ylim=(-1.05, 1.05), color="#303030", lw=1.0, alpha=0.55)
for k, v in enumerate(pentagon_vertices):
    draw_point(ax, v, label=str(k + 1), color="#111111", size=35, dx=0.025, dy=0.025)
set_equal_window(ax, (-1.08, 1.08), (-1.08, 1.08), "Hyperbolic: five right angles")
ax.text(-1.02, -1.0, f"max |l_i^T B l_(i+1)| = {max(pentagon_residuals):.1e}", fontsize=9)
model_comparison_path = save_figure(fig, "cayley-klein-model-comparison.png")

MODEL_COMPARISON_ROWS = [
    {"model": "Euclidean", "absolute_data": "double line at infinity with complex circular points", "visible_rule": "finite orthogonal slopes multiply to -1", "figure_target": "negative reciprocal slope", "numeric_check": slope_checks["euclidean_product"], "takeaway": "the projective test specializes to the usual quarter-turn rule"},
    {"model": "pseudo-Euclidean", "absolute_data": "double line at infinity with real isotropic points", "visible_rule": "finite orthogonal slopes multiply to +1", "figure_target": "inverse slope reflected across a slope-one direction", "numeric_check": slope_checks["pseudo_product"], "takeaway": "self-orthogonal directions appear at slopes 1 and -1"},
    {"model": "hyperbolic Klein disk", "absolute_data": "real nondegenerate conic", "visible_rule": "adjacent lines can all be B-orthogonal in a pentagon", "figure_target": "five right angles inside the disk", "numeric_check": max(pentagon_residuals), "takeaway": "right-angle behavior differs from Euclidean polygon intuition"},
    {"model": "rank-one dual conic", "absolute_data": "dual object B=q q^T", "visible_rule": "a line through q is orthogonal to every line", "figure_target": "implicit altitude choices become underdetermined", "numeric_check": CONSTRUCTIVE_IMPLICIT_CHECKS["implicit_concurrence_det"], "takeaway": "constructive and implicit formulations can diverge"},
]
model_table_path = save_table(MODEL_COMPARISON_ROWS, ARTIFACT_ROOT, "tables", "model-comparison-samples.csv")

MODEL_COMPARISON_CHECKS = {
    "slope_checks": slope_checks,
    "pentagon_orthogonality_residuals": pentagon_residuals,
    "pentagon_vertex_radii": pentagon_radii,
    "all_vertices_inside_absolute": all(radius < 1 for radius in pentagon_radii),
}
MODEL_COMPARISON_CHECKS

## 4. Midpoints and Angle Bisectors Are Dual Operations

The midpoint formula gives two points, not one. In a Euclidean specialization one is the ordinary midpoint and the other is the point at infinity on the joining line. In a nondegenerate disk model both can be finite, but the second may lie outside the visible absolute. The invariant relation is harmonic: $(p,q;m_+,m_-)=-1$.

Angle bisectors are the dual construction. For two lines $l$ and $g$, the two bisectors are formed by the same square-root pattern with $\Theta_{ll}=l^TBl$ and $\Theta_{gg}=g^TBg$. The pair of bisectors is itself $B$-orthogonal.

In [ ]:
P_mid = hpoint(-0.20, 0.10)
Q_mid = hpoint(0.75, 0.20)
lam = math.sqrt(omega(Q_mid))
mu = math.sqrt(omega(P_mid))
M_plus = hnormalize(lam * P_mid + mu * Q_mid)
M_minus = hnormalize(lam * P_mid - mu * Q_mid)
t_plus = mu / (lam + mu)
t_minus = -mu / (lam - mu)
midpoint_cross_ratio_error = abs(cross_ratio(0, 1, t_plus, t_minus) + 1)
midpoint_join = normalize_line(join(P_mid, Q_mid))

S = hpoint(0.12, -0.08)
l_angle = normalize_line(line_through_point_direction(S, (math.cos(math.radians(22)), math.sin(math.radians(22)))))
g_angle = normalize_line(line_through_point_direction(S, (math.cos(math.radians(118)), math.sin(math.radians(118)))))
theta_l = theta(l_angle)
theta_g = theta(g_angle)
coef_g = math.sqrt(abs(theta_l))
coef_l = math.sqrt(abs(theta_g))
bis_plus = normalize_line(coef_g * g_angle + coef_l * l_angle)
bis_minus = normalize_line(coef_g * g_angle - coef_l * l_angle)
bisector_orthogonality_residual = abs(ck_orthogonality(bis_plus, bis_minus))
bisector_incidence_residuals = [abs(float(S @ bis_plus)), abs(float(S @ bis_minus))]

fig, axes = plt.subplots(1, 2, figsize=(12.0, 5.4))
ax = axes[0]
draw_unit_circle(ax)
draw_line(ax, midpoint_join, xlim=(-0.55, 2.75), ylim=(-0.55, 1.2), color="#303030", lw=1.4)
for point, label, color, marker in [
    (P_mid, "p", "#111111", "o"),
    (Q_mid, "q", "#111111", "o"),
    (M_plus, "m+", "#1d4ed8", "s"),
    (M_minus, "m-", "#b3261e", "D"),
]:
    draw_point(ax, point, label=label, color=color, marker=marker, size=52)
set_equal_window(ax, (-0.55, 2.75), (-0.55, 1.2), "Two midpoints on one projective line")
ax.text(-0.50, -0.48, f"cross-ratio error from -1: {midpoint_cross_ratio_error:.1e}", fontsize=9)

ax = axes[1]
draw_unit_circle(ax)
for line, label, color, style in [
    (l_angle, "l", "#1d4ed8", "-"),
    (g_angle, "g", "#1d4ed8", "-"),
    (bis_plus, "a+", "#b3261e", "--"),
    (bis_minus, "a-", "#d97706", "--"),
]:
    draw_line(ax, line, xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), label=label, color=color, lw=2.0, ls=style)
draw_point(ax, S, label="p", color="#111111", size=52)
set_equal_window(ax, (-1.08, 1.08), (-1.08, 1.08), "Dual formula: two angle bisectors")
ax.legend(loc="upper right", fontsize=8)
ax.text(-1.02, -1.0, f"|a+^T B a-| = {bisector_orthogonality_residual:.1e}", fontsize=9)
midpoint_bisector_path = save_figure(fig, "midpoints-angle-bisectors-duality.png")

MIDPOINT_BISECTOR_CHECKS = {
    "midpoint_cross_ratio_error": float(midpoint_cross_ratio_error),
    "midpoint_parameters": {"t_plus": float(t_plus), "t_minus": float(t_minus)},
    "bisector_orthogonality_residual": float(bisector_orthogonality_residual),
    "bisector_incidence_residuals": bisector_incidence_residuals,
}
MIDPOINT_BISECTOR_CHECKS

### Symbolic Sanity Scaffold

The source proofs are short because the right projective form makes the algebra cancel. The next cell records two small cancellations explicitly: the harmonic relation for the midpoint pair, and the orthogonality of the two angle bisectors.

In [ ]:
lam_sym, mu_sym, theta_l_sym, theta_g_sym = sp.symbols("lambda mu Theta_l Theta_g", nonzero=True)


def det2(u, v):
    return sp.Matrix([[u[0], v[0]], [u[1], v[1]]]).det()


p_sym = sp.Matrix([1, 0])
q_sym = sp.Matrix([0, 1])
m_plus_sym = sp.Matrix([lam_sym, mu_sym])
m_minus_sym = sp.Matrix([lam_sym, -mu_sym])
harmonic_identity = sp.simplify(
    det2(p_sym, m_plus_sym) * det2(q_sym, m_minus_sym)
    + det2(p_sym, m_minus_sym) * det2(q_sym, m_plus_sym)
)
bisector_identity = sp.simplify(theta_l_sym * theta_g_sym - theta_g_sym * theta_l_sym)
assert harmonic_identity == 0
assert bisector_identity == 0

SYMBOLIC_CHECKS = {
    "midpoint_harmonic_identity": str(harmonic_identity),
    "midpoint_harmonic_inputs": "Use line coordinates p=(1,0), q=(0,1), m+=(lambda,mu), m-=(lambda,-mu); the bracket expression [p,m+][q,m-]+[p,m-][q,m+] cancels to zero.",
    "bisector_orthogonality_identity": str(bisector_identity),
    "bisector_orthogonality_inputs": "Use a+=sqrt(Theta_l) g + sqrt(Theta_g) l and a-=sqrt(Theta_l) g - sqrt(Theta_g) l; mixed bilinear terms cancel and the remaining Theta_l*Theta_g - Theta_g*Theta_l is zero.",
}
symbolic_path = save_json(SYMBOLIC_CHECKS, ARTIFACT_ROOT, "checks", "symbolic-identities.json")
SYMBOLIC_CHECKS

## 5. Trigonometry as a Projective Identity

The chapter's law of sines says that the Euclidean, elliptic, and hyperbolic formulas are specializations of one projective statement. For triangle vertices $P,Q,R$ with opposite side lines

$$p=QR, \quad q=RP, \quad r=PQ,$$

define

$$\Psi_{PQ}=\frac{(P\times Q)^T B(P\times Q)}{(P^TAP)(Q^TAQ)}, \qquad
\Psi^*_{pq}=\frac{(p\times q)^T A(p\times q)}{(p^TBp)(q^TBq)}.$$

The theorem asserts

$$\frac{\Psi_{PQ}}{\Psi^*_{pq}}=\frac{\Psi_{QR}}{\Psi^*_{qr}}=\frac{\Psi_{RP}}{\Psi^*_{rp}}.$$

The static diagram places the three ratios on a single triangle. The Plotly lab that follows moves the third vertex and lets the learner inspect that the equality is not a property of one lucky picture.

In [ ]:
P_tri = hpoint(-0.34, -0.20)
Q_tri = hpoint(0.58, -0.10)
R_tri = hpoint(0.05, 0.56)
p_line = normalize_line(join(Q_tri, R_tri))
q_line = normalize_line(join(R_tri, P_tri))
r_line = normalize_line(join(P_tri, Q_tri))

psi_values = {
    "Psi_PQ": psi_point(P_tri, Q_tri),
    "Psi_QR": psi_point(Q_tri, R_tri),
    "Psi_RP": psi_point(R_tri, P_tri),
    "Psi_star_pq": psi_line(p_line, q_line),
    "Psi_star_qr": psi_line(q_line, r_line),
    "Psi_star_rp": psi_line(r_line, p_line),
}
projective_sine_ratios = [
    psi_values["Psi_PQ"] / psi_values["Psi_star_pq"],
    psi_values["Psi_QR"] / psi_values["Psi_star_qr"],
    psi_values["Psi_RP"] / psi_values["Psi_star_rp"],
]
projective_sine_ratio_spread = float(max(projective_sine_ratios) - min(projective_sine_ratios))

A_general = np.array([[1.7, 0.2, -0.1], [0.2, 1.3, 0.15], [-0.1, 0.15, 1.1]])
B_general = np.array([[1.1, -0.25, 0.05], [-0.25, 1.6, 0.12], [0.05, 0.12, 1.4]])
general_ratios = [
    psi_point(P_tri, Q_tri, A_general, B_general) / psi_line(p_line, q_line, A_general, B_general),
    psi_point(Q_tri, R_tri, A_general, B_general) / psi_line(q_line, r_line, A_general, B_general),
    psi_point(R_tri, P_tri, A_general, B_general) / psi_line(r_line, p_line, A_general, B_general),
]
general_ratio_spread = float(max(general_ratios) - min(general_ratios))

fig, ax = plt.subplots(figsize=(7.5, 7.0))
draw_unit_circle(ax)
ax.plot([P_tri[0], Q_tri[0], R_tri[0], P_tri[0]], [P_tri[1], Q_tri[1], R_tri[1], P_tri[1]], color="#111111", lw=2.0)
for point, label in [(P_tri, "P"), (Q_tri, "Q"), (R_tri, "R")]:
    draw_point(ax, point, label=label, color="#111111", size=58)
for line, label in [(p_line, "p"), (q_line, "q"), (r_line, "r")]:
    draw_line(ax, line, xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), label=label, color="#666666", lw=1.1, alpha=0.75)
ratio_text = "\n".join([
    f"Psi(PQ)/Psi*(pq) = {projective_sine_ratios[0]: .6f}",
    f"Psi(QR)/Psi*(qr) = {projective_sine_ratios[1]: .6f}",
    f"Psi(RP)/Psi*(rp) = {projective_sine_ratios[2]: .6f}",
    f"spread = {projective_sine_ratio_spread:.1e}",
])
ax.text(-1.05, -1.02, ratio_text, fontsize=9, family="monospace", bbox={"facecolor": "white", "edgecolor": "#cccccc", "alpha": 0.9})
set_equal_window(ax, (-1.1, 1.1), (-1.1, 1.1), "Projective law of sines check")
ax.legend(loc="upper right", fontsize=8)
law_of_sines_path = save_figure(fig, "projective-law-of-sines-check.png")

LAW_OF_SINES_CHECKS = {
    "psi_values": psi_values,
    "projective_sine_ratios": projective_sine_ratios,
    "projective_sine_ratio_spread": projective_sine_ratio_spread,
    "general_matrix_ratios": general_ratios,
    "general_matrix_ratio_spread": general_ratio_spread,
}
LAW_OF_SINES_CHECKS

In [ ]:
def law_ratios_for_R(R_point):
    p_side = normalize_line(join(Q_tri, R_point))
    q_side = normalize_line(join(R_point, P_tri))
    r_side = normalize_line(join(P_tri, Q_tri))
    return [
        psi_point(P_tri, Q_tri) / psi_line(p_side, q_side),
        psi_point(Q_tri, R_point) / psi_line(q_side, r_side),
        psi_point(R_point, P_tri) / psi_line(r_side, p_side),
    ]


sweep_rows = []
for t in np.linspace(0.0, 1.0, 13):
    R_sample = hpoint(-0.24 + 0.56 * t, 0.34 + 0.18 * math.sin(math.pi * t) + 0.12 * t)
    ratios = law_ratios_for_R(R_sample)
    sweep_rows.append({
        "t": float(t),
        "R_x": float(R_sample[0]),
        "R_y": float(R_sample[1]),
        "ratio_PQ_over_angle_R": float(ratios[0]),
        "ratio_QR_over_angle_P": float(ratios[1]),
        "ratio_RP_over_angle_Q": float(ratios[2]),
        "spread": float(max(ratios) - min(ratios)),
    })

sweep_table_path = save_table(sweep_rows, ARTIFACT_ROOT, "tables", "law-of-sines-sweep.csv")
max_sweep_spread = max(row["spread"] for row in sweep_rows)

circle_theta = np.linspace(0, 2 * np.pi, 260)
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.cos(circle_theta), y=np.sin(circle_theta), mode="lines", name="absolute", line={"color": "#333333", "width": 2}))
for idx, row in enumerate(sweep_rows):
    x_vals = [P_tri[0], Q_tri[0], row["R_x"], P_tri[0]]
    y_vals = [P_tri[1], Q_tri[1], row["R_y"], P_tri[1]]
    fig.add_trace(go.Scatter(
        x=x_vals,
        y=y_vals,
        mode="lines",
        line={"color": "rgba(179, 38, 30, 0.28)", "width": 1.4},
        name="sample triangles" if idx == 0 else None,
        showlegend=(idx == 0),
        hoverinfo="skip",
    ))
fig.add_trace(go.Scatter(
    x=[row["R_x"] for row in sweep_rows],
    y=[row["R_y"] for row in sweep_rows],
    mode="markers+lines",
    name="moving R",
    marker={"size": 9, "color": [row["spread"] for row in sweep_rows], "colorscale": "Viridis", "showscale": True, "colorbar": {"title": "spread"}},
    customdata=np.array([[row["t"], row["ratio_PQ_over_angle_R"], row["ratio_QR_over_angle_P"], row["ratio_RP_over_angle_Q"], row["spread"]] for row in sweep_rows]),
    hovertemplate="t=%{customdata[0]:.2f}<br>ratio 1=%{customdata[1]:.8f}<br>ratio 2=%{customdata[2]:.8f}<br>ratio 3=%{customdata[3]:.8f}<br>spread=%{customdata[4]:.2e}<extra></extra>",
))
fig.add_trace(go.Scatter(x=[P_tri[0], Q_tri[0]], y=[P_tri[1], Q_tri[1]], mode="markers+text", text=["P", "Q"], textposition="top center", name="fixed P,Q", marker={"size": 11, "color": "#111111"}))
fig.update_layout(
    title="Projective law-of-sines lab: move R and inspect the invariant ratio",
    xaxis={"scaleanchor": "y", "range": [-1.08, 1.08], "zeroline": False},
    yaxis={"range": [-1.08, 1.08], "zeroline": False},
    width=820,
    height=680,
    margin={"l": 40, "r": 40, "t": 70, "b": 40},
)
lab_html_path = HTML / "projective-law-of-sines-lab.html"
fig.write_html(lab_html_path, include_plotlyjs=True, full_html=True)

LAB_CHECKS = {
    "sweep_rows": len(sweep_rows),
    "max_sweep_spread": float(max_sweep_spread),
    "html_artifact": book_relative(lab_html_path),
    "table_artifact": book_relative(sweep_table_path),
}
LAB_CHECKS

## Proof Dependency Map

This chapter has many familiar theorem names, but the computational flow is small. The map is a proof scaffold: read each arrow as a projective translation supplying the input for the next construction.

In [ ]:
G = nx.DiGraph()
node_labels = {
    "cross_ratio": "cross-ratio\nmeasurement",
    "orthogonality": "$l^T Bm=0$\northogonality",
    "pole": "pole $Bl$",
    "altitudes": "constructive\naltitudes",
    "degeneracy": "implicit\ndegeneracy",
    "models": "Euclidean / pseudo /\nhyperbolic contrasts",
    "midpoints": "two midpoint\nformula",
    "harmonic": "harmonic\npairs",
    "bisectors": "dual angle\nbisectors",
    "psi": "$\\Psi$, $\\Psi^*$\nmeasurements",
    "sines": "projective law\nof sines",
}
G.add_nodes_from(node_labels)
G.add_edges_from([
    ("cross_ratio", "orthogonality"),
    ("orthogonality", "pole"),
    ("pole", "altitudes"),
    ("orthogonality", "degeneracy"),
    ("orthogonality", "models"),
    ("cross_ratio", "midpoints"),
    ("midpoints", "harmonic"),
    ("harmonic", "bisectors"),
    ("cross_ratio", "psi"),
    ("psi", "sines"),
])
pos = {
    "cross_ratio": (0, 2),
    "orthogonality": (1.7, 2.4),
    "pole": (3.3, 2.7),
    "altitudes": (5.1, 2.7),
    "degeneracy": (3.3, 1.8),
    "models": (5.1, 1.8),
    "midpoints": (1.7, 0.9),
    "harmonic": (3.3, 0.9),
    "bisectors": (5.1, 0.9),
    "psi": (1.7, 0.0),
    "sines": (3.7, 0.0),
}
fig, ax = plt.subplots(figsize=(11.2, 5.9))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=15, width=1.5, edge_color="#777777")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2100, node_color="#e0f2fe", edgecolors="#075985", linewidths=1.3)
nx.draw_networkx_labels(G, pos, labels=node_labels, ax=ax, font_size=9)
ax.set_axis_off()
ax.set_title("Chapter 22 proof and construction dependencies", fontsize=12)
proof_map_path = save_figure(fig, "chapter-22-proof-dependency-map.png")

PROOF_MAP_CHECKS = {"nodes": G.number_of_nodes(), "edges": G.number_of_edges(), "is_dag": nx.is_directed_acyclic_graph(G)}
PROOF_MAP_CHECKS

## Artifact Gallery

These direct links keep the notebook readable before execution. The same files are displayed inline by the following code cell.

![Orthogonality through poles](../../artifacts/chapter-22-cayley-klein-geometries-at-work/figures/orthogonality-pole-polar-altitudes.png)

![Constructive versus implicit altitudes](../../artifacts/chapter-22-cayley-klein-geometries-at-work/figures/constructive-vs-implicit-altitudes.png)

![Cayley-Klein model comparison](../../artifacts/chapter-22-cayley-klein-geometries-at-work/figures/cayley-klein-model-comparison.png)

![Midpoints and angle bisectors](../../artifacts/chapter-22-cayley-klein-geometries-at-work/figures/midpoints-angle-bisectors-duality.png)

![Projective law of sines](../../artifacts/chapter-22-cayley-klein-geometries-at-work/figures/projective-law-of-sines-check.png)

The interactive lab is saved as [projective-law-of-sines-lab.html](../../artifacts/chapter-22-cayley-klein-geometries-at-work/html/projective-law-of-sines-lab.html).

In [ ]:
STATIC_ARTIFACTS = [
    orthogonality_path,
    constructive_implicit_path,
    model_comparison_path,
    midpoint_bisector_path,
    law_of_sines_path,
    proof_map_path,
]
for artifact in STATIC_ARTIFACTS:
    display_artifact(artifact, width=820)
display_artifact(lab_html_path, width="100%", height=720)

## Applied Lab

Use the Plotly lab as a small experiment rather than a finished picture.

1. Hover over the moving $R$ samples and compare the three projective sine-law ratios.
2. Notice that the color scale is almost constant because it encodes only the residual spread.
3. Change the sample path in the code cell if you want a sharper stress test. Avoid points on the absolute and avoid side lines tangent to the dual conic, since those are exactly the denominators excluded in the theorem.

The lab also records a CSV sweep so the equality can be audited without opening the HTML file.

In [ ]:
pd.DataFrame(sweep_rows).head()

## Final Sanity Checks

The final cell verifies both the mathematical checks and the artifact contract: paths are book-local, files exist, raster artifacts are nonblank, and the key residuals are below tolerance.

In [ ]:
ALL_ARTIFACTS = [
    *STATIC_ARTIFACTS,
    lab_html_path,
    model_table_path,
    sweep_table_path,
    storyboard_path,
    symbolic_path,
]
assert_artifacts(ALL_ARTIFACTS, min_size=256)

raster_stats = []
for path in STATIC_ARTIFACTS:
    stats = image_stats(path)
    stats["path"] = book_relative(path)
    raster_stats.append(stats)
assert all(item["pixel_std"] > 1.0 for item in raster_stats)

assert max(ORTHOGONALITY_CHECKS["orthogonality_residuals"].values()) < 1e-9
assert ORTHOGONALITY_CHECKS["altitude_concurrence_det"] < 1e-9
assert ORTHOGONALITY_CHECKS["orthocenter_incidence_residual"] < 1e-9
assert max(CONSTRUCTIVE_IMPLICIT_CHECKS["implicit_orthogonality_residuals"]) < 1e-9
assert CONSTRUCTIVE_IMPLICIT_CHECKS["implicit_concurrence_det"] > 1e-3
assert abs(MODEL_COMPARISON_CHECKS["slope_checks"]["euclidean_product"] + 1) < 1e-12
assert abs(MODEL_COMPARISON_CHECKS["slope_checks"]["pseudo_product"] - 1) < 1e-12
assert max(MODEL_COMPARISON_CHECKS["pentagon_orthogonality_residuals"]) < 1e-9
assert MODEL_COMPARISON_CHECKS["all_vertices_inside_absolute"]
assert MIDPOINT_BISECTOR_CHECKS["midpoint_cross_ratio_error"] < 1e-9
assert MIDPOINT_BISECTOR_CHECKS["bisector_orthogonality_residual"] < 1e-9
assert LAW_OF_SINES_CHECKS["projective_sine_ratio_spread"] < 1e-9
assert LAW_OF_SINES_CHECKS["general_matrix_ratio_spread"] < 1e-9
assert LAB_CHECKS["max_sweep_spread"] < 1e-9
assert PROOF_MAP_CHECKS["is_dag"]

visual_checks = {
    "chapter": 22,
    "all_files_exist": all(path.exists() for path in ALL_ARTIFACTS),
    "cross_ratio_error": MIDPOINT_BISECTOR_CHECKS["midpoint_cross_ratio_error"],
    "raster_artifacts": raster_stats,
    "html_artifact": book_relative(lab_html_path),
    "table_artifacts": [book_relative(model_table_path), book_relative(sweep_table_path)],
    "numeric_checks": {
        "max_altitude_orthogonality_residual": max(ORTHOGONALITY_CHECKS["orthogonality_residuals"].values()),
        "altitude_concurrence_det": ORTHOGONALITY_CHECKS["altitude_concurrence_det"],
        "implicit_nonconcurrence_det": CONSTRUCTIVE_IMPLICIT_CHECKS["implicit_concurrence_det"],
        "max_pentagon_orthogonality_residual": max(MODEL_COMPARISON_CHECKS["pentagon_orthogonality_residuals"]),
        "bisector_orthogonality_residual": MIDPOINT_BISECTOR_CHECKS["bisector_orthogonality_residual"],
        "projective_sine_ratio_spread": LAW_OF_SINES_CHECKS["projective_sine_ratio_spread"],
        "general_matrix_ratio_spread": LAW_OF_SINES_CHECKS["general_matrix_ratio_spread"],
        "max_law_of_sines_sweep_spread": LAB_CHECKS["max_sweep_spread"],
    },
    "visual_count": len(STATIC_ARTIFACTS) + 1,
}
visual_checks_path = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")

final_sanity = {
    "chapter": 22,
    "source_span": "printed pages 423-442; PDF pages 445-464",
    "artifacts": all_book_relative(ALL_ARTIFACTS),
    "checks": visual_checks["numeric_checks"],
    "symbolic_checks": SYMBOLIC_CHECKS,
    "raster_artifacts": raster_stats,
    "all_files_exist": visual_checks["all_files_exist"],
}
final_sanity_path = save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")

Markdown(f"Saved `{book_relative(visual_checks_path)}` and `{book_relative(final_sanity_path)}`.")

## Takeaways

- Orthogonality in a Cayley-Klein geometry is a pole-polar statement: $l^T Bm=0$.
- Constructive theorems survive more reliably than implicit ones because a nonzero pole selects a specific line; rank-deficient absolutes can leave too much freedom.
- Euclidean, pseudo-Euclidean, and hyperbolic examples share one algebraic test but display different angle behavior.
- Midpoints and angle bisectors are dual square-root constructions, and each naturally comes as a pair.
- The law of sines is best read here as a projective identity among $\Psi$ and $\Psi^*$ ratios; classical trigonometric formulas are specializations.